# move_detection: stable frames + move inference (M5, optional)

Move detection from a stream of warped top-down board images. Pipeline per move:
1. `StableFrameDetector` watches frame-to-frame differences and fires once the board is
   still again after a period of motion (the player's hand came in and left).
2. The stable frame is cut into 64 crops and classified -> (64, 13) probabilities.
3. `infer_move()` scores every legal move of the previous position (plus "no move")
   against those probabilities and returns the most likely one. Because candidates come
   from python-chess, castling, en passant and promotion need no special cases.

Status: design skeleton; every function below is a TODO. Evaluated on recorded games by
`run_move_detection.ipynb`.

Milestone: M5 (optional).

**TODO** (see `docs/M5_TODO.md`)
- [ ] `frame_difference` and the `StableFrameDetector` state machine
- [ ] `placement_log_likelihood`, `infer_move`, `changed_squares`
- [ ] record games (video/frames + PGN)
- [ ] tune `StableFrameDetector` / `frame_difference`
- [ ] build the `board_proba` callable (homography warp -> crops -> M3 model)
- [ ] `MoveTracker.process_frame`
- [ ] `evaluate_on_game`
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

import chess
import numpy as np

%run ./labels.ipynb
# board_assembly.ipynb (M4) is only needed for the optional cross-check in MoveTracker;
# %run it in the run notebook if you use it.

## Stable frames
State machine IDLE -> MOTION -> SETTLING -> STABLE -> IDLE on the mean absolute frame
difference.

In [ ]:
class State(str, Enum):
    IDLE = "idle"          # board static, nothing happening
    MOTION = "motion"      # large frame differences (hand over the board)
    SETTLING = "settling"  # differences small again, waiting for settle_frames in a row
    STABLE = "stable"      # emitted for exactly one frame after settling; then back to IDLE


def frame_difference(a: np.ndarray, b: np.ndarray) -> float:
    """Mean absolute difference between two frames, in [0, 255] intensity units.

    Color frames are averaged to grayscale first. Different shapes -> ValueError.
    """
    # Design:
    # - Cast to float first (uint8 subtraction wraps around).
    # - Ideas to try: downsample or Gaussian blur first to suppress sensor noise / flicker;
    #   a per-square max instead of a global mean so a small hand at the board edge is not
    #   averaged away.
    # TODO: implement
    raise NotImplementedError


class StableFrameDetector:
    """State machine IDLE -> MOTION -> SETTLING -> STABLE -> IDLE.

    Feed frames with update(frame); it returns True exactly once per motion episode, on the
    frame where the board has been still for `settle_frames` consecutive frames after at
    least `min_motion_frames` frames of motion. `stable_frame` / `reference_frame` hold the
    frames after and before the episode.

    Defaults are guesses: tune diff_threshold / settle_frames on recorded video (plot
    frame_difference over time for a few games), and decide whether to add an explicit hand
    detector (e.g. skin-color mask or "occupancy outside the start/end squares") so slow
    hand movements are not mistaken for a still board.
    """

    def __init__(self, diff_threshold: float = 4.0, settle_frames: int = 10,
                 min_motion_frames: int = 2) -> None:
        self.diff_threshold = diff_threshold
        self.settle_frames = settle_frames
        self.min_motion_frames = min_motion_frames
        self.reset()

    def reset(self) -> None:
        """Back to IDLE with no frames seen."""
        # Design: state; prev_frame; reference_frame (last still frame before motion);
        # stable_frame (frame at which STABLE fired); last_diff; motion / still counters.
        # TODO: implement
        raise NotImplementedError

    def update(self, frame: np.ndarray) -> bool:
        """Feed one frame; True exactly on the frame where STABLE fires."""
        # Design (transitions on moving = frame_difference(prev, frame) > diff_threshold):
        # - First frame: just store it (also as reference).
        # - IDLE/STABLE: moving -> MOTION; still -> IDLE and update reference_frame.
        # - MOTION: count motion frames; still after >= min_motion_frames -> SETTLING;
        #   still after fewer -> back to IDLE (a one-frame blip from noise / flicker).
        # - SETTLING: moving -> back to MOTION; still -> count; settle_frames in a row ->
        #   STABLE, remember stable_frame, return True.
        # Why a reference frame: comparing before/after the episode tells which squares changed.
        # TODO: implement
        raise NotImplementedError

## Move inference
Score = sum over squares of log P(square has the label the candidate position implies).
`ambiguity_margin` is in nats: 2.0 means the best candidate is ~7x more likely than the
runner-up.

In [ ]:
@dataclass
class MoveResult:
    move: chess.Move | None        # None = board judged unchanged
    san: str | None                # SAN in the previous position, None for "no move"
    score: float                   # log-likelihood of the best candidate
    runner_up: chess.Move | None   # second-best candidate (None may mean "no move")
    margin: float                  # score(best) - score(runner_up), >= 0
    ambiguous: bool                # margin below threshold -> ask for confirmation / wait
    ranking: list[tuple[chess.Move | None, float]] = field(default_factory=list, repr=False)


def placement_log_likelihood(labels: Sequence[str], log_probs: np.ndarray) -> float:
    """Sum over squares of log P(square has this label); log_probs is (64, 13)."""
    # Design: map labels to class indices (CLASS_TO_INDEX), pick one entry per row with
    # fancy indexing, sum. Treats squares as independent.
    # TODO: implement
    raise NotImplementedError


def infer_move(prev_board: chess.Board, square_probs_after: np.ndarray, *,
               include_null: bool = True, ambiguity_margin: float = 2.0,
               eps: float = 1e-6) -> MoveResult:
    """Pick the legal move that best explains the new per-square probabilities.

    Inputs: prev_board (position before the move), square_probs_after (64, 13), a1 first,
    labels.CLASSES order. Returns: MoveResult (best move, runner-up, margin, ambiguous flag).
    """
    # Design:
    # - Shape check -> ValueError. log-probs with probabilities clipped to [eps, 1].
    # - Candidates = every legal move of prev_board (+ "no move" if include_null).
    # - For each candidate, the board it produces gives one label per square; score it by
    #   placement_log_likelihood. Why this works: castling / en passant / promotion need no
    #   special cases, because python-chess produces the resulting board (push / pop on a copy).
    # - Sort; margin = best - runner-up; ambiguous = margin < ambiguity_margin.
    #   No candidates at all (game over, include_null False) -> ambiguous empty result.
    # Decisions for you: eps for log(0); what margin counts as ambiguous (in nats); unknown
    # side to move (mid-game start) and two-ply candidates when a stable frame was missed.
    # TODO: implement
    raise NotImplementedError


def changed_squares(prev_labels: Sequence[str], new_labels: Sequence[str]) -> list[str]:
    """Names of squares whose label differs, in square-index order (a1 first)."""
    # TODO: implement (ValueError unless both have 64 labels)
    raise NotImplementedError

## Tracking a game

In [ ]:
ProbaFn = Callable[[np.ndarray], np.ndarray]  # warped frame -> (64, 13) probabilities


@dataclass
class MoveEvent:
    frame_index: int
    result: MoveResult
    fen_after: str


class MoveTracker:
    """Detector + classifier + infer_move over a frame stream, maintaining the game board.

    `board_proba` maps a warped top-down frame to (64, 13) probabilities; build it from
    capture.ipynb's extract_all_crops + an M3 model's predict_proba.
    """

    def __init__(self, board_proba: ProbaFn, start_board: chess.Board | None = None,
                 detector: StableFrameDetector | None = None,
                 ambiguity_margin: float = 2.0) -> None:
        # Design: store board_proba and ambiguity_margin; own a *copy* of start_board
        # (default chess.Board()) so the caller's board is not mutated; default detector
        # StableFrameDetector(); events list; frame counter.
        # TODO: implement
        raise NotImplementedError

    def process_frame(self, frame: np.ndarray) -> MoveEvent | None:
        """Feed one warped frame; return a MoveEvent when a move was recognized."""
        # Design:
        # - Count the frame; only act when detector.update(frame) fires.
        # - probs = board_proba(frame); result = infer_move(self.board, probs, ...).
        # - result.move is None -> hand passed over / piece adjusted; no event.
        # - result.ambiguous -> do not push a guess: wait for the next stable frame or flag
        #   for the user/robot.
        # - Else push onto self.board, record MoveEvent(frame_index, result, board.fen()).
        # - Optional cross-check with assemble(probs) from board_assembly.ipynb.
        # TODO: implement
        raise NotImplementedError

    def run(self, frames: Iterable[np.ndarray]) -> list[MoveEvent]:
        """Process a whole stream (e.g. frames read from a video and warped by homography)."""
        # TODO: implement
        raise NotImplementedError


def evaluate_on_game(pgn_path: str | Path, frames_dir: str | Path,
                     board_proba: ProbaFn | None = None, **tracker_kwargs) -> dict:
    """Replay one recorded game through MoveTracker and score it against its PGN.

    Inputs: the game's PGN (ground truth) and a directory of warped frames in time order
    (e.g. data/crops/<session>/frames/ or frames extracted from the game video).

    Returns a metrics dict (run_move_detection.ipynb writes it to
    results/<run_name>/move_metrics.json):
        n_moves             plies in the PGN
        move_accuracy       fraction of PGN plies recognized correctly, aligned in order
                            (a missed or wrong move counts as an error; after an error the
                            tracker is resynced to the true position so one mistake does
                            not cascade - report the unsynced "game completed" flag too)
        missed              PGN plies with no detected move
        spurious            detected moves with no corresponding ply
        ambiguous           detections flagged ambiguous
        special_moves       accuracy restricted to castling / en passant / promotion plies
        game_completed      True if the tracker reproduced the full game without resync
    """
    # Design:
    # - chess.pgn.read_game for the truth; frames sorted by name; build the tracker.
    # - Align detected moves with mainline moves in order; resync after an error.
    # - Special moves: board.is_castling / is_en_passant / move.promotion on the true move.
    # TODO: implement
    raise NotImplementedError

## Self-check
(was tests/test_move_detection.py)

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# Helper: one_hot(board, noise) -> (64, 13) probabilities of a board with optional uniform noise.
# - infer_move on the board after the move, for each case: start + e2e4 -> "e4";
#   "r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1" + e1g1 -> "O-O" and e1c1 -> "O-O-O";
#   "rnbqkbnr/1pp1pppp/p7/3pP3/8/8/PPPP1PPP/RNBQKBNR w KQkq d6 0 3" + e5d6 -> "exd6" (en passant);
#   "8/P7/8/8/8/8/k6K/8 w - - 0 1" + a7a8q -> "a8=Q+". Each not ambiguous, margin > 0.
# - Unchanged board -> move None, san None, not ambiguous.
# - noise=0.5 still picks g1f3.
# - (64, 12) probabilities -> ValueError.
# - changed_squares before/after e2e4 == ["e2", "e4"].
# - StableFrameDetector(diff_threshold=4, settle_frames=5) on synthetic 64x64 frames: 5 still
#   frames (noise sigma 0.5), 4 random "hand" frames, 12 still frames with one patch changed
#   -> fires once at index 14; stable_frame is frame 14; reference_frame is frame 4 (the last
#   still frame before the hand); state back to IDLE afterwards.
# - evaluate_on_game with a board_proba returning one-hot probabilities of the true position
#   gives move_accuracy == 1.0 and reports special-move accuracy (needs a short recorded or
#   rendered game).
print("move_detection: self-check not written yet (see plan above)")